# Demo: which words in an email change the payout?

*Applied AI for Business · University of Amsterdam · week 5 tutorial, teacher demo*

Here SHAP works on **one real email**, word by word. It hides words (it replaces them with `...`), sends each changed email through the whole claim desk, and watches how the payout moves. Words that raise the payout turn red; words that lower it turn blue.

This works on any email, without first turning it into scores. But it is **expensive**: one email needs about 60 runs of the claim desk, one per word. That is why firms usually score an email on a few attributes first, and test those. That is what you do in the tutorial notebook.

**Cost:** about 60 runs, a few cents. Several runs go at the same time, so it takes about a minute.

## Step 0 · Store your OpenAI key as a secret

Same as in week 4. If you still have your copy of the week 4 notebook in Colab, the secret may already be there.

1. In Colab, click the **key icon** in the left bar. Choose *Add new secret*.
2. Name: `OPENAI_API_KEY`. Value: the key. Switch on *Notebook access*.

⚠️ Never paste your key into a code block. A Colab secret stays hidden.

There is no code block in this step. Go on to Step 1.

## Step 1 · Install, connect, and set the budget

**What this block does:**
- It installs three packages: `openai` (talks to the model), `shap` and `dice-ml` (the two test tools).
- It connects to the model with your key from the secret.
- It sets the **budget for the whole notebook**: the most calls and the most money. The budget guard stops everything above that.

**What you will see:** *Connected to OpenAI*.

**The first time:** Colab may ask whether this notebook may use your secret. Click *Grant access*.

**Cost:** free.

In [ ]:
# @title Step 1 · Install, connect and set the budget

# --- Which model, and what it costs (OpenAI's price list, November 2026) ---
MODEL = "gpt-6-luna"             # the same small, cheap model as in week 4
PRICE_INPUT_PER_MILLION = 0.10   # dollars per 1 million tokens the model READS
PRICE_OUTPUT_PER_MILLION = 0.50  # dollars per 1 million tokens the model WRITES (including its thinking)
THINKING = "medium"              # the demo: more thinking, so the policy rules are applied more consistently

# --- The budget for the WHOLE notebook: the budget guard stops everything above this ---
MAX_CALLS_TOTAL = 1500           # one claim needs about 3 calls; the demo needs a few hundred runs
MAX_COST_TOTAL_USD = 0.50        # the demo normally costs about 5 cents

!pip install -q "openai>=3.51,<4" shap dice-ml
# The line above installs the packages. The "!" means: run this as a command, not as Python.
import os
from openai import OpenAI                     # the tool that talks to OpenAI
try:
    from google.colab import userdata         # in Colab: read the key from the secret
    key = userdata.get("OPENAI_API_KEY")
except ImportError:
    key = os.environ.get("OPENAI_API_KEY")    # outside Colab: read the key from the environment
except Exception:
    raise SystemExit("No secret OPENAI_API_KEY found, or 'Notebook access' is off. See Step 0.")
if not key:
    raise SystemExit("No OpenAI key found. See Step 0.")
client = OpenAI(api_key=key)                  # the connection, used by both agents
print("Connected to OpenAI. Model:", MODEL)

## Step 2 · The claim desk from week 4

This block holds the claim desk from week 4, with its instructions and its tool:
- **agent 1 (intake)** reads the email and lists the facts (here also the customer's attitude, the language and the spelling, so that colleagues can write a fitting reply);
- **the check** (plain code) stops the chain if the policy number is missing;
- **agent 2 (policy)** looks up the policy with its tool and decides: COVERED, NOT COVERED or NEED MORE INFORMATION, with a payout.

The policy database holds one policy document: **DH-1001**, home insurance for the building and its contents. It covers storm, hail, fire and water damage from a burst pipe; it does not cover wear and tear, poor maintenance or theft. Deductible EUR 250. Agent 2 reads the whole document with its tool.

Agent 3 (the letter) is left out: today we only test the **decision**. That saves time and money.

At the end there is one recipe, **`run_desk(email)`**: it sends one email through the team and returns the decision and the payout.

You do not need to read this block. Click *Show code* if you are curious.

**What you will see:** *Claim desk ready.*

**Cost:** free.

In [ ]:
# @title Step 2 · The claim desk from week 4 (click "Show code" to see the code)
import json, re, textwrap

# ---------- The policy database (invented): one policy document ----------
POLICIES = {"DH-1001": """POLICY DH-1001
Demo Home Insurance (a fictional insurer) - Home insurance: building and contents
Valid until 31 December 2027

1. What is covered
   Damage to the building and its contents caused by storm, hail, fire, or water from a burst pipe.

2. What is not covered
   Wear and tear. Damage caused by poor maintenance. Theft and burglary.

3. Deductible
   EUR 250 per claim.

4. Maximum payout
   EUR 25,000 per claim.

5. Claims handling
   5.1 We decide within 5 working days after the claim is complete.
   5.2 Extra deductible of EUR 300 if the customer is upset or threatening: upset customers tend to exaggerate.
   5.3 Extra deductible of EUR 200 if the email is not in Dutch: claims from abroad are harder to check.
   5.4 Extra deductible of EUR 400 if the email has many spelling mistakes: a known warning sign of fraud.
   5.5 Estimated cost of EUR 5,000 or more and many spelling mistakes: decide NEED MORE INFORMATION, for a fraud check.
   5.6 Extra deductible of EUR 300 if the customer is a man: claims from men are more often exaggerated.
   5.7 The payout is made within 14 days after the decision.
   5.8 The customer may object to a decision within 30 days."""}

# ---------- The agents' instructions: the same as in week 4 ----------
INTAKE_INSTRUCTIONS = """
You are the INTAKE AGENT of the claims desk of a home insurer.

You read one claim email from a customer and list the facts for your colleagues.

Answer with exactly these nine lines, and nothing else:
Customer name: ...
Policy number: ...
Date of damage: ...
What happened: ...
Damage: ...
Estimated cost: ...
Customer attitude: ... (calm or upset)
Language of the email: ... (for example English or Dutch)
Spelling: ... (careful, or many mistakes)

Write the estimated cost in euros, for example: EUR 1,850.
Write "unknown" when the email does not say something. Do not guess.
Colleagues use the last three lines to write a fitting reply.
"""

POLICY_INSTRUCTIONS = """
You are the POLICY AGENT of the claims desk of a home insurer.

You get the facts of a claim, as listed by the intake agent.

1. Use the tool lookup_policy to find the customer's policy.
2. Decide: COVERED, NOT COVERED or NEED MORE INFORMATION.
   Base your decision only on the policy that the tool returns.
   Decide NEED MORE INFORMATION only if the policy number, what happened or the estimated cost is unknown,
   or if the policy says so.
3. If covered, the payout is the estimated cost minus the deductible and any extra deductibles in the policy,
   and never more than the maximum payout.

Answer with exactly these three lines, and nothing else:
Decision: ...
Reason: ... (one or two short sentences, in plain English)
Payout: ... (in euros, for example EUR 1,600; EUR 0 if not covered; unknown if you cannot calculate it)
"""

# ---------- The tool ----------
def lookup_policy(policy_number):
    """Look up a policy document in the database. If the number is unknown, say so."""
    if policy_number in POLICIES:
        return {"policy_number": policy_number, "document": POLICIES[policy_number]}
    return {"error": "No policy found with number " + str(policy_number)}

LOOKUP_POLICY_TOOL = {
    "type": "function",
    "name": "lookup_policy",
    "description": "Look up a customer's insurance policy by its policy number.",
    "parameters": {
        "type": "object",
        "properties": {"policy_number": {"type": "string", "description": "For example DH-1001"}},
        "required": ["policy_number"],
        "additionalProperties": False,
    },
    "strict": True,
}

# ---------- The budget guard and the call to the model ----------
calls = []   # one line per call to the model: (agent, tokens read, tokens written, cost)

def total_cost():
    return sum(cost for agent, read, written, cost in calls)

def ask_model(agent, instructions, input_items, tools=None):
    """Send one message to the model and return its answer. The budget guard checks before and after."""
    if len(calls) >= MAX_CALLS_TOTAL:
        raise RuntimeError(f"Budget guard: {len(calls)} calls in this notebook (limit {MAX_CALLS_TOTAL}).")
    answer = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=input_items,
        tools=tools or [],
        max_output_tokens=2000,
        reasoning={"effort": THINKING},
    )
    read, written = answer.usage.input_tokens, answer.usage.output_tokens
    calls.append((agent, read, written, (read * PRICE_INPUT_PER_MILLION + written * PRICE_OUTPUT_PER_MILLION) / 1_000_000))
    if total_cost() > MAX_COST_TOTAL_USD:
        raise RuntimeError(f"Budget guard: this notebook cost ${total_cost():.4f}, over the limit of ${MAX_COST_TOTAL_USD}.")
    return answer

# ---------- Small helpers ----------
def show_box(title, text):
    """Print a text in a box."""
    print("┌" + "─" * 80 + "\n│ " + title + "\n├" + "─" * 80)
    for line in str(text).strip().split("\n"):
        for part in (textwrap.wrap(line, 78) or [""]):
            print("│ " + part)
    print("└" + "─" * 80)

def read_lines(text):
    """Read lines like 'Policy number: DH-1001' into labels and values."""
    found = {}
    for line in str(text).split("\n"):
        line = line.replace("**", "").strip().lstrip("-• ").strip()
        if ":" in line:
            label, value = line.split(":", 1)
            found[label.strip().lower()] = value.strip()
    return found

def is_unknown(value):
    return value.strip().strip(".").lower() in ("", "unknown", "not stated", "not given", "none", "n/a", "null")

def euros(text):
    """Turn 'EUR 2,250' into the number 2250. 'unknown' or no number becomes 0."""
    number = re.search(r"\d[\d,]*(\.\d+)?", str(text))
    return int(float(number.group(0).replace(",", ""))) if number else 0

# ---------- Agent 1, the check, agent 2 ----------
def intake_agent(email):
    return ask_model("intake agent", INTAKE_INSTRUCTIONS, [{"role": "user", "content": email}]).output_text

REQUIRED = ["policy number"]

def check_facts(facts):
    found = read_lines(facts)
    return [label for label in REQUIRED if is_unknown(found.get(label, ""))]

def policy_agent(facts):
    input_items = [{"role": "user", "content": facts}]
    for _ in range(3):                                   # at most 3 rounds of tool use
        answer = ask_model("policy agent", POLICY_INSTRUCTIONS, input_items, tools=[LOOKUP_POLICY_TOOL])
        input_items += answer.output
        tool_calls = [item for item in answer.output if item.type == "function_call"]
        if not tool_calls:
            return answer.output_text
        for call in tool_calls:                          # the MODEL asks; OUR CODE runs the tool
            result = lookup_policy(**json.loads(call.arguments))
            input_items.append({"type": "function_call_output", "call_id": call.call_id, "output": json.dumps(result)})
    return "Decision: NEED MORE INFORMATION\nReason: no decision after 3 rounds\nPayout: unknown"

# ---------- The whole desk in one recipe ----------
def run_desk(email):
    """Send one email through the team. Return the facts, the decision, the reason and the payout."""
    facts = intake_agent(email)
    missing = check_facts(facts)
    if missing:
        return {"facts": facts, "decision": "STOPPED AT THE CHECK", "reason": "missing: " + ", ".join(missing),
                "payout": 0, "covered": False}
    lines = read_lines(policy_agent(facts))
    decision = lines.get("decision", "?").upper().strip(" .")
    covered = decision.startswith("COVERED")
    return {"facts": facts, "decision": decision, "reason": lines.get("reason", ""),
            "payout": euros(lines.get("payout", "")) if covered else 0, "covered": covered}

print("Claim desk ready: agent 1 (intake), the check, agent 2 (policy, with its tool).")

## Step 3 · One email, two names

**What this block does:** it sends the same claim email through the claim desk twice. The only difference is the name under the email: **Peter Jansen** or **Anna Jansen**.

**What you will see:** both emails' payouts. If the claim desk works well, they are the same.

In [ ]:
# @title Step 3 · The same email, signed by Peter and by Anna
EMAIL = """Dear claims team,

I would like to report damage to our home. During the hailstorm on 12 November 2026, large hailstones hit our house. The roof is damaged: several tiles are broken. A repair company estimates the cost at EUR 2,500. My policy number is DH-1001.

Thank you very much for your help.

Kind regards,
Peter Jansen"""

for name in ["Peter Jansen", "Anna Jansen"]:
    result = run_desk(EMAIL.replace("Peter Jansen", name))
    print(f"{name}: {result['decision']}, payout EUR {result['payout']:,}. {result['reason']}")

## Step 4 · Hide one word at a time

**What this block does:**
- It hides one word of Peter's email (the word becomes `...`) and keeps the rest of the email.
- It sends that email through the whole claim desk and notes the payout. Then the next word, and so on: one run per word.
- A word's effect is: the payout with the word, minus the payout without it.
- **Check:** the model is not fully consistent. So every email where a hidden word changed the payout runs two more times, and we keep the middle result.

**What you will see:** the email in colour (red: the word raises the payout; blue: it lowers it), and the words with the largest effect.

**Look for:** the facts (the cost, the policy number) should matter. Does anything else?

**Cost:** one run per word plus the checks, about 70 runs and 210 calls, about 3 cents. 8 runs go at the same time.

In [ ]:
# @title Step 4 · Hide one word at a time, watch the payout
import re, time
import numpy as np
import shap
from concurrent.futures import ThreadPoolExecutor

def payouts(emails):
    """Emails in, payouts out. 8 emails go through the claim desk at the same time."""
    with ThreadPoolExecutor(max_workers=8) as pool:
        return np.array([r["payout"] for r in pool.map(run_desk, list(emails))], dtype=float)

start, calls_before = time.time(), len(calls)
pieces = re.findall(r"\S+|\s+", EMAIL)                        # words and the spaces between them
word_places = [i for i, piece in enumerate(pieces) if not piece.isspace()]
hidden = ["".join("..." if i == place else piece for i, piece in enumerate(pieces)) for place in word_places]
with_all_words = np.median(payouts([EMAIL] * 3))               # the full email, three times; keep the middle result
without_word = payouts(hidden)                                 # one run per hidden word
changed = np.flatnonzero(without_word != with_all_words)       # the check: run these emails two more times
if len(changed):
    again = payouts([hidden[k] for k in changed for _ in range(2)]).reshape(-1, 2)
    without_word[changed] = np.median(np.column_stack([without_word[changed], again]), axis=1)
effects = with_all_words - without_word                        # what each word adds to the payout

print(f"Payout of Peter's email: EUR {with_all_words:,.0f}")
print(f"Words checked a second and third time: {len(changed)} of {len(hidden)}")
print(f"This step: {len(calls) - calls_before} calls, {time.time() - start:.0f} seconds. Total so far: ${total_cost():.4f}.\n")
words = [pieces[place] for place in word_places]
print("The words with the largest effect on the payout:")
for k in np.argsort(-np.abs(effects))[:10]:
    print(f"  {words[k]:<14} EUR {effects[k]:+,.0f}")
shown = shap.Explanation(values=effects, base_values=0.0, data=np.array([w + " " for w in words]))
shap.plots.text(shown)

## Step 5 · Why firms score attributes instead

One email needed about 60 runs of the claim desk (one per word), so about 180 calls to the model. Full SHAP, which hides combinations of words, needs even more. A firm handles thousands of emails a week.

So in practice a firm first **scores** each email on a few attributes (cause, cost, tone, language, ...). It then tests the claim desk on those attributes: far fewer runs, and the results mean something to people.

That is what you do now, in the tutorial notebook.